# User Defined Functions (UDF) - Databricks

## O que são UDFs?

UDFs (User Defined Functions) no Spark permitem criar funções customizadas para estender as capacidades do Spark SQL e PySpark quando as funções nativas não atendem às necessidades do negócio.

## Tipos de UDF

### 1. SQL UDF (Recomendada)
```sql
CREATE OR REPLACE FUNCTION catalog.schema.udf_name(param_name data_type)
RETURNS return_type
RETURN expression;
```

**Vantagens:**
- Melhor performance (otimização do Catalyst)
- Não serializa/deserializa dados
- Recomendada pela Databricks

### 2. Python UDF
```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def my_udf(column_value):
    return column_value.upper()

# Ou registrar para SQL
spark.udf.register("my_udf", my_udf)
```

**Desvantagens:**
- Overhead de serialização Python ↔ JVM
- Performance inferior às SQL UDFs


## Comparação de Performance
```
SQL UDF > Python UDF
```

## Boas Práticas

1. **Prefira SQL UDFs** quando possível
2. Evite Python UDFs tradicionais em produção
3. Registre UDFs no Unity Catalog para reutilização
4. Documente o propósito e parâmetros da UDF

## Exemplo Prático: UDF no Unity Catalog
```sql
-- Criar UDF permanente
CREATE OR REPLACE FUNCTION techdados.bronze.format_cpf(cpf STRING)
RETURNS STRING
RETURN CONCAT(
    SUBSTRING(cpf, 1, 3), '.',
    SUBSTRING(cpf, 4, 3), '.',
    SUBSTRING(cpf, 7, 3), '-',
    SUBSTRING(cpf, 10, 2)
);

-- Usar a UDF
SELECT format_cpf('12345678901') as cpf_formatado;
```

## Documentação Oficial

- [User-defined functions - SQL](https://learn.microsoft.com/pt-br/azure/databricks/sql/language-manual/sql-ref-functions-udf)
- [User-defined functions - Python](https://learn.microsoft.com/pt-br/azure/databricks/udf/)

## Quando Usar

✅ **Use UDF quando:**
- Lógica de negócio específica não disponível em funções nativas
- Reutilização de código entre queries
- Transformações complexas customizadas

❌ **Evite UDF quando:**
- Funções nativas do Spark já resolvem o problema
- Performance é crítica (prefira SQL nativo)
- Pode ser resolvido com operações de DataFrame

In [0]:
%sql
CREATE OR REPLACE FUNCTION techdados.silver.classify_customer_seniority(nome_param DATE)
RETURNS STRING
COMMENT 'Classifica o cliente por tempo de cadastro: Novo (<90 dias), Recente (<1 ano), Estabelecido (<2 anos), Veterano (2+ anos)'
RETURN 
  CASE 
    -- Clientes com menos de 90 dias (3 meses) de cadastro
    WHEN DATEDIFF(CURRENT_DATE(), nome_param) < 90 THEN 'Novo'
    
    -- Clientes entre 90 dias e 1 ano de cadastro
    WHEN DATEDIFF(CURRENT_DATE(), nome_param) < 365 THEN 'Recente'
    
    -- Clientes entre 1 e 2 anos de cadastro
    WHEN DATEDIFF(CURRENT_DATE(), nome_param) < 730 THEN 'Estabelecido'
    
    -- Clientes com mais de 2 anos de cadastro
    ELSE 'Veterano'
  END;

In [0]:
%sql
SELECT 
*,
techdados.silver.classify_customer_seniority(created_at) AS customer_seniority
FROM techdados.silver.customers